# 06a — Logistic Regression — SCAFFOLD

**Milestone 3 · Task 7.** Build the Logistic Regression classifier and record its
performance against the 54.7% majority-class floor.

Logistic Regression fits a weighted sum of the features and squashes it
through a sigmoid to produce a probability. Each feature gets one
coefficient describing how much it moves the log-odds of success.

**It needs scaling.** `log_goal` ranges roughly 0-17 while
`name_has_exclaim` is 0 or 1; without standardisation the regularisation
penalty falls on them unevenly. `make_models()` already wraps it in a
`StandardScaler` pipeline, which also ensures the scaler is fitted on
training folds only during cross-validation.

**Expect collinearity.** `log_goal`, `log_goal_per_day`,
`log_goal_per_tier` and `log_goal_vs_subcat_median` all encode versions
of the same quantity, so individual coefficients may be unstable even
when the model as a whole is fine.

### How this fits with the other two notebooks

Three people, three models, three notebooks, run in parallel. Each notebook
writes its own CSV to `reports/`, and Task 8 concatenates all three into one
comparison table.

**Two rules so the results line up:**

1. Build `X` and `y` with `build_model_matrix()` — never by hand.
2. Report through `evaluate()` — never write your own metric code. It returns
   a one-row DataFrame so the rows from all three notebooks stack cleanly.

Do not edit anything in `src/`. If you think a shared module needs changing,
raise it with the repo owner first.

## Setup

Run this first. On Colab it clones the repo; locally it does nothing.

In [ ]:
# Works both locally and on Google Colab.
# On Colab this clones the repo; locally it does nothing.
import sys, os
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "State-Street-1B-kickstarter-crowdfunding-recommendation-engine"
    GITHUB_URL = f"https://github.com/YOUR-ORG/{REPO}.git"   # <- edit this
    if not os.path.exists(REPO):
        os.system(f"git clone -q {GITHUB_URL}")
    os.chdir(f"{REPO}/notebooks")

sys.path.insert(0, "..")
print("Colab" if IN_COLAB else "local", "| cwd:", Path.cwd().name)

In [ ]:
import pandas as pd
import numpy as np

from src.features import build_model_matrix
from src.modeling import make_models, align_columns, random_split, chronological_split
from src.evaluation import evaluate, majority_class_baseline, confusion_frame

pd.set_option("display.width", 220)
EXTRA = ["log_city_project_count", "log_goal_vs_subcat_median"]

# The parquet splits are gitignored, so they will not exist on a fresh clone.
# Rebuild them if missing -- about 40 seconds.
if not Path("../data/m3_random_train.parquet").exists():
    print("Splits not found. Rebuilding from the raw data ...")
    from src.loading import load_raw
    from src.cleaning import clean
    from src.features import build_features, add_location_frequency, add_relative_goal

    feat = build_features(clean(load_raw("../data/DSI_kickstarterscrape_dataset_csv.zip"),
                                verbose=False)[0])

    def _prep(tr, te):
        tr, city = add_location_frequency(tr)
        tr, goal = add_relative_goal(tr)
        te, _ = add_location_frequency(te, reference=city)
        te, _ = add_relative_goal(te, reference=goal)
        return tr, te

    for (a, b), names in [(_prep(*random_split(feat)), ("m3_random_train", "m3_random_test")),
                          (_prep(*chronological_split(feat)), ("m3_chrono_train", "m3_chrono_test"))]:
        a.to_parquet(f"../data/{names[0]}.parquet", index=False)
        b.to_parquet(f"../data/{names[1]}.parquet", index=False)
    print("Splits written.")

SPLITS = {
    "random": (pd.read_parquet("../data/m3_random_train.parquet"),
               pd.read_parquet("../data/m3_random_test.parquet")),
    "chronological": (pd.read_parquet("../data/m3_chrono_train.parquet"),
                      pd.read_parquet("../data/m3_chrono_test.parquet")),
}
for n, (tr, te) in SPLITS.items():
    print(f"{n:15s} train {len(tr):,}  test {len(te):,}")

## The floor

Predicting the majority class for every row. Any model that fails to beat this has learned nothing, which is why reporting accuracy without this number is misleading.

Note the recall of 1.0: always guessing "successful" catches every success and is wrong about every failure. The floor still scores 0.71 on F1, so F1 alone is not enough either.

In [ ]:
tr, te = SPLITS["random"]
floor = majority_class_baseline(tr["success"], te["success"])
floor.round(4)

## Your task

Fit **Logistic Regression** on all four combinations:

| feature_set | split |
|---|---|
| `launch_only` | `random` |
| `launch_only` | `chronological` |
| `plus_campaign` | `random` |
| `plus_campaign` | `chronological` |

`launch_only` is the primary feature set — it excludes `updates`, which is
unavailable at campaign launch. `plus_campaign` adds it back so the cost of
that exclusion can be measured in Task 8 rather than argued about.

**Expect roughly 65–72% on `launch_only`, not the 80% in the brief.** That is
the correct result, not a mistake: the benchmark was produced with `updates`
included.

### The pattern for one combination

```python
train, test = SPLITS["random"]
X_train, y_train = build_model_matrix(train, "launch_only", extra_numeric=EXTRA)
X_test,  y_test  = build_model_matrix(test,  "launch_only", extra_numeric=EXTRA)
X_test = align_columns(X_train, X_test)   # one-hot columns must match exactly

model = make_models()["Logistic Regression"]
model.fit(X_train, y_train)

evaluate(model, X_test, y_test,
         label="Logistic Regression", feature_set="launch_only", split="random")
```

Loop that over both feature sets and both splits, collect the rows, and
`pd.concat` them.

In [ ]:
results = []

for split_name, (train, test) in SPLITS.items():
    for fs in ["launch_only", "plus_campaign"]:
        # TODO: build X_train / y_train and X_test / y_test with
        #       build_model_matrix(), then align_columns() the test matrix
        # TODO: model = make_models()["Logistic Regression"]  then model.fit(...)
        # TODO: results.append(evaluate(model, X_test, y_test,
        #                               label="Logistic Regression",
        #                               feature_set=fs, split=split_name))
        pass

# res = pd.concat(results, ignore_index=True)
# res.round(4)

## Inspect the results

Once the loop runs, look at the primary configuration — `launch_only` on the `random` split. That is the one that answers the project's actual question.

In [ ]:
# TODO: filter res to feature_set == "launch_only" and split == "random"
# TODO: compare its accuracy against the floor above
# TODO: print the lift over the floor

## Confusion matrix

`confusion_frame(model, X_test, y_test)` returns a labelled matrix. The two off-diagonal cells are different harms:

- **False positive** — a creator is told to launch, and the campaign fails
- **False negative** — a creator is discouraged from a campaign that would have succeeded

Which matters more is a question Milestone 4 has to answer.

In [ ]:
# TODO: refit on launch_only / random and show confusion_frame(...)

## Save your results

Task 8 reads this file. **Use exactly this filename** or the comparison will not pick it up.

In [ ]:
# TODO: uncomment once `res` exists
# res.to_csv("../reports/m3_task7_logistic.csv", index=False)
# print("Wrote ../reports/m3_task7_logistic.csv")

## Write-up

Fill this in when the numbers are available.

- **Accuracy on `launch_only` / `random`:** ___ (floor is 54.7%)
- **Lift over the floor:** ___
- **What `updates` is worth:** ___ (launch_only vs plus_campaign)
- **Chronological vs random:** ___

### Things to watch with this model

- Coefficients are the real prize here. On standardised features each one
  reads as "one standard deviation more of this moves the log-odds by X",
  which converts directly into Milestone 4 advice.
- If you see a convergence warning, raise `max_iter` rather than ignoring it.